# 02 — Optimizers from scratch: GD, momentum, RMSProp, Adam, AdamW, schedules

**Goal:** understand *why* Adam is the default, what momentum buys on an ill-conditioned problem, how the convergence
rate depends on the condition number κ, and what "decoupled weight decay" means. Every optimizer you write is checked
step-by-step against `torch.optim`. Your ENS labs called `optim.Adam(...)`; here you become it.
**Time:** ~3 h. **Level:** 2 / 5. **Prerequisite:** notebook 01.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch

rng = np.random.default_rng(0)

def check(cond, msg):
    assert cond, "❌ " + msg
    print("✅", msg)

## Part 1 — The optimizer interface

Every optimizer holds its own state and updates a parameter vector in place: `opt.step(x, g)` with `x` and `g`
NumPy arrays of the same shape. Formulas (exactly PyTorch's conventions, so the checks can match):

| Optimizer | Update |
|---|---|
| SGD | `x ← x − lr·g` |
| Momentum (heavy ball) | `v ← μ·v + g`, `x ← x − lr·v` (v starts at 0) |
| RMSProp | `s ← α·s + (1−α)·g²`, `x ← x − lr·g / (√s + ε)` |
| Adam | `m ← β₁m + (1−β₁)g`, `v ← β₂v + (1−β₂)g²`, `m̂ = m/(1−β₁ᵗ)`, `v̂ = v/(1−β₂ᵗ)`, `x ← x − lr·m̂/(√v̂ + ε)` |
| AdamW | first `x ← x·(1 − lr·λ)`, then the Adam step |

In [ ]:
class SGD:
    def __init__(self, lr):
        self.lr = lr
    def step(self, x, g):
        x -= self.lr * g

class Momentum:
    def __init__(self, lr, mu=0.9):
        # TODO: your code here
        raise NotImplementedError
    def step(self, x, g):
        # TODO: your code here
        raise NotImplementedError

class RMSProp:
    def __init__(self, lr, alpha=0.99, eps=1e-8):
        # TODO: your code here
        raise NotImplementedError
    def step(self, x, g):
        # TODO: your code here
        raise NotImplementedError

class Adam:
    def __init__(self, lr, betas=(0.9, 0.999), eps=1e-8, weight_decay=0.0):
        """weight_decay here is AdamW's decoupled decay (0 = plain Adam)."""
        # TODO: your code here
        raise NotImplementedError
    def step(self, x, g):
        # TODO: your code here
        raise NotImplementedError

The check runs 100 steps of yours and of `torch.optim` on the same non-quadratic function and compares trajectories.
(Note the momentum detail: PyTorch's first step sets `v = g`, which is what `μ·0 + g` gives too.)

In [ ]:
def f_torch(x):                     # a bumpy 3-D function
    return (x[0] - 1) ** 2 + 10 * (x[1] + 0.5) ** 2 + torch.sin(3 * x[2]) + 0.1 * x[2] ** 2

def grad_np(x):
    xt = torch.tensor(x, requires_grad=True)
    f_torch(xt).backward()
    return xt.grad.numpy()

def run_mine(opt, x0, n=100):
    x = x0.copy(); traj = [x.copy()]
    for _ in range(n):
        opt.step(x, grad_np(x)); traj.append(x.copy())
    return np.array(traj)

def run_torch(make_opt, x0, n=100):
    xt = torch.tensor(x0.copy(), requires_grad=True); opt = make_opt([xt]); traj = [x0.copy()]
    for _ in range(n):
        opt.zero_grad(); f_torch(xt).backward(); opt.step(); traj.append(xt.detach().numpy().copy())
    return np.array(traj)

x0 = np.array([3.0, 2.0, 1.0])
pairs = {
    "SGD":      (SGD(0.02),                         lambda p: torch.optim.SGD(p, lr=0.02)),
    "Momentum": (Momentum(0.02, 0.9),               lambda p: torch.optim.SGD(p, lr=0.02, momentum=0.9)),
    "RMSProp":  (RMSProp(0.01),                     lambda p: torch.optim.RMSprop(p, lr=0.01)),
    "Adam":     (Adam(0.05),                        lambda p: torch.optim.Adam(p, lr=0.05)),
    "AdamW":    (Adam(0.05, weight_decay=0.1),      lambda p: torch.optim.AdamW(p, lr=0.05, weight_decay=0.1)),
}
for name, (mine, theirs) in pairs.items():
    check(np.allclose(run_mine(mine, x0), run_torch(theirs, x0), atol=1e-8), f"{name} matches torch.optim step by step")

## Part 2 — Ill-conditioning: why plain GD is slow

`f(x) = ½ xᵀAx` with `A = diag(1, κ)`, κ = 50. The curvature is 50× larger along one axis.

In [ ]:
kappa = 50.0
A = np.diag([1.0, kappa])
quad = lambda x: 0.5 * x @ A @ x
quad_grad = lambda x: A @ x

def run(opt, x0, n):
    x = x0.copy(); xs = [x.copy()]
    for _ in range(n):
        opt.step(x, quad_grad(x)); xs.append(x.copy())
    return np.array(xs)

**✍️ Predict before running.** Gradient descent with the largest stable learning rate: will it zig-zag along x₂, crawl along x₁, or both? Will momentum (μ = 0.9) help? Will Adam behave differently, and why (think: per-coordinate scaling)?

> Prediction:
>
> Confidence: [1] / [2] / [3]

In [ ]:
x0 = np.array([10.0, 1.0])
runs = {"GD lr=0.039": run(SGD(0.039), x0, 100),
        "Momentum lr=0.02 μ=0.7": run(Momentum(0.02, 0.7), x0, 100),
        "Adam lr=0.3": run(Adam(0.3), x0, 100)}
g1, g2 = np.meshgrid(np.linspace(-11, 11, 200), np.linspace(-2, 2, 200))
fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
axes[0].contour(g1, g2, 0.5 * (g1 ** 2 + kappa * g2 ** 2), levels=20, cmap="Greys", linewidths=0.5)
for name, xs in runs.items():
    axes[0].plot(xs[:, 0], xs[:, 1], ".-", ms=3, label=name)
    axes[1].semilogy([quad(x) for x in xs], label=name)
axes[0].legend(); axes[1].set_xlabel("step"); axes[1].set_ylabel("f"); axes[1].legend(); plt.show()

### Convergence rates, measured

Theory for a quadratic with eigenvalues in `[μ_min, L]`, κ = L / μ_min:
- GD with the best fixed step `lr = 2/(L + μ_min)` contracts the error by `(κ−1)/(κ+1)` per step.
- Heavy-ball momentum with the best `lr = 4/(√L + √μ_min)²` and `μ = ((√κ−1)/(√κ+1))²` contracts by `(√κ−1)/(√κ+1)`.

For κ = 50 that is 0.961 vs 0.752: momentum needs ~7× fewer steps. Measure it.

In [ ]:
def empirical_rate(xs):
    """Geometric-mean contraction of ||x_t|| over the last half of the run (the optimum is 0)."""
    # TODO: your code here
    raise NotImplementedError

L, m = kappa, 1.0
lr_gd = None; lr_hb = None; mu_hb = None   # TODO: the optimal values from the formulas above
r_gd = empirical_rate(run(SGD(lr_gd), x0, 200))
r_hb = empirical_rate(run(Momentum(lr_hb, mu_hb), x0, 200))
print(f"GD: measured {r_gd:.4f}  theory {(kappa - 1) / (kappa + 1):.4f}")
print(f"HB: measured {r_hb:.4f}  theory {(np.sqrt(kappa) - 1) / (np.sqrt(kappa) + 1):.4f}")
check(abs(r_gd - (kappa - 1) / (kappa + 1)) < 0.005, "GD rate matches (κ−1)/(κ+1)")
check(abs(r_hb - (np.sqrt(kappa) - 1) / (np.sqrt(kappa) + 1)) < 0.03, "momentum rate matches (√κ−1)/(√κ+1)")

This is the real reason feature standardisation matters for linear models and normalisation layers matter for deep
nets: they reduce κ.

## Part 3 — Mini-batch SGD: noise and batch size

Logistic regression on 5 000 points, 20 features. Full-batch gradient vs mini-batches.

In [ ]:
from sklearn.datasets import make_classification
Xm, ym = make_classification(n_samples=5000, n_features=20, n_informative=10, random_state=0)
Xm = (Xm - Xm.mean(0)) / Xm.std(0)
sig = lambda z: 1 / (1 + np.exp(-z))
loss_full = lambda w: np.mean(np.logaddexp(0, Xm @ w) - ym * (Xm @ w))

def minibatch_grad(w, idx):
    """Gradient of the mean BCE on the rows idx."""
    # TODO: your code here
    raise NotImplementedError

def train_sgd(batch_size, lr, epochs=5, seed=0):
    """Shuffle each epoch, loop over mini-batches, one SGD step per batch. Return full-data loss after every step."""
    # TODO: your code here
    raise NotImplementedError

**✍️ Predict before running.** Same number of epochs (5), lr = 0.5. Batch 8 vs batch 256 vs full batch (5000): which ends with the lowest loss? Which curve is noisiest? How many steps does each take?

> Prediction:
>
> Confidence: [1] / [2] / [3]

In [ ]:
res = {b: train_sgd(b, 0.5) for b in [8, 256, 5000]}
for b, L_ in res.items():
    plt.plot(np.linspace(0, 5, len(L_)), L_, label=f"batch {b} ({len(L_)} steps)", lw=1)
plt.xlabel("epoch"); plt.ylabel("full-data loss"); plt.ylim(0.3, 0.7); plt.legend(); plt.show()
check(len(res[8]) == 5 * 625 and len(res[5000]) == 5, "number of steps per epoch is n / batch_size")
check(res[256][-1] < res[5000][-1], "at equal epochs, mini-batches beat full batch (many more updates)")
check(np.std(np.diff(res[8][-200:])) > np.std(np.diff(res[256][-15:])), "small batches are noisier")

## Part 4 — Learning-rate schedules: warmup + cosine

Transformers are trained with a linear warmup followed by cosine decay. You'll use it in notebook 11.

In [ ]:
def warmup_cosine(step, total, warmup, lr_max, lr_min=0.0):
    """step in [0, total). Linear from lr_max/warmup (step 0) to lr_max (step warmup-1), i.e. lr_max*(step+1)/warmup;
    then cosine from lr_max down to lr_min at step total-1."""
    # TODO: your code here
    raise NotImplementedError

In [ ]:
T, W = 1000, 100
lrs = [warmup_cosine(s, T, W, 1e-3, 1e-5) for s in range(T)]
check(np.isclose(lrs[0], 1e-5) and np.isclose(lrs[W - 1], 1e-3), "warmup goes linearly up to lr_max")
check(np.isclose(lrs[W], 1e-3) and np.isclose(lrs[-1], 1e-5), "cosine starts at lr_max and ends at lr_min")
check(all(a >= b for a, b in zip(lrs[W:], lrs[W + 1:])), "monotone decay after warmup")
plt.figure(figsize=(5, 2.5)); plt.plot(lrs); plt.xlabel("step"); plt.ylabel("lr"); plt.show()

Why warm up at all? Adam's `v̂` is estimated from very few gradients at the start, so the first steps can be huge in
directions with tiny historical gradients. Write one sentence in your own words in the log.

## Part 5 — Adam vs AdamW: L2 penalty is not weight decay for Adam

With SGD, adding `λ‖w‖²/2` to the loss and decaying weights by `(1 − lr·λ)` are the same thing. With Adam they are not:
the L2 gradient `λw` gets divided by `√v̂`, so weights with large gradients are barely regularised.

**✍️ Predict before running.** Two parameters: one gets large gradients (±100, alternating sign every step, so they cancel out on average), one tiny ones (±0.01). With 'Adam + L2 in the gradient' vs 'AdamW', which parameter shrinks faster in each case?

> Prediction:
>
> Confidence: [1] / [2] / [3]

In [ ]:
# TODO: g_scale = np.array([100.0, 0.01]); start both weights at 1.0; for 500 steps (t = 0..499) the "data" gradient
# is (-1)**t * g_scale. Variant A: Adam(lr=1e-2), feed data gradient + lam * w (L2 penalty, lam = 1e-1).
# Variant B: Adam(lr=1e-2, weight_decay=lam), feed the data gradient only. Final weights in wA, wB.
print("Adam + L2 :", np.round(wA, 3)); print("AdamW     :", np.round(wB, 3))
check(abs(wB[0] - wB[1]) < 0.02, "AdamW decays both weights at the same rate, whatever their gradient scale")
check(wA[0] > wB[0] + 0.03, "Adam + L2 barely regularises the weight with large gradients (λw is divided by √v̂ ≈ 100)")
check(abs(wA[1]) < 0.5 * abs(wB[1]), "and pushes the small-gradient weight to 0 far harder")

## Debrief → `learning/LOG.md`

1. The Adam update from memory, with bias correction, and why bias correction is needed at step 1.
2. GD vs momentum rates as functions of κ; what normalisation does to κ.
3. Batch size: noise vs number of updates per epoch.
4. AdamW vs Adam + L2, in one sentence.
5. **Still shaky.**